# Inspect North River Feature Store V1

This notebook is a local QA pass for the Version 1 external-forcing feature store. It checks that the processed file exists, has the expected hourly structure, and contains plausible rainfall/tide features.

It does not train a model, create target-flow columns, or assume `influent_flow_mgd` exists.

## 1. Basic File Check

**Purpose:** Before doing deeper validation, this confirms that the processed feature-store file exists and can be loaded. If this fails, the next step is to rebuild the feature store from the repo root with `python3 src/build_feature_store_v1.py`.

In [ ]:
# Load the processed feature store. The notebook supports being run from the repo root or from notebooks/.
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import polars as pl

candidate_paths = [
    Path("data/processed/north_river_feature_store_v1.parquet"),
    Path("../data/processed/north_river_feature_store_v1.parquet"),
]
FEATURE_STORE_PATH = next((path for path in candidate_paths if path.exists()), candidate_paths[0])

# Stop early with a useful message if the generated output is missing.
if not FEATURE_STORE_PATH.exists():
    raise FileNotFoundError(
        "Feature store not found. From the repo root, run: "
        "python3 src/build_feature_store_v1.py"
    )

df = pl.read_parquet(FEATURE_STORE_PATH)
print("file exists:", FEATURE_STORE_PATH.exists())
print("shape:", df.shape)
print("columns:")
for col in df.columns:
    print("-", col)

display(df.head(3))
display(df.tail(3))

## 2. Time-Axis Validation

**Purpose:** The hourly backbone is the foundation of this dataset. The feature store should have a complete hourly sequence from `2015-01-01 00:00` through `2024-12-31 23:00`, with no duplicate or missing timestamps. If the time axis is wrong, every lag and rolling feature becomes harder to trust.

In [ ]:
from datetime import datetime, timedelta

start = df.select(pl.col("timestamp").min()).item()
end = df.select(pl.col("timestamp").max()).item()
expected_start = datetime(2015, 1, 1, 0)
expected_end = datetime(2024, 12, 31, 23)
# Expected row count for a simple complete hourly backbone.
expected_rows = int((expected_end - expected_start).total_seconds() / 3600) + 1
duplicate_timestamps = df.height - df.select("timestamp").n_unique()

actual_timestamps = set(df["timestamp"].to_list())
missing = []
current = expected_start
while current <= expected_end:
    if current not in actual_timestamps:
        missing.append(current)
    current += timedelta(hours=1)

# Timestamp differences should be dominated by one-hour steps.
delta_counts = (
    df.select(pl.col("timestamp").diff().alias("delta"))
    .group_by("delta")
    .len()
    .sort("len", descending=True)
)

print("start timestamp:", start)
print("end timestamp:", end)
print("row count:", df.height)
print("expected row count:", expected_rows)
print("duplicate timestamps:", duplicate_timestamps)
print("missing timestamps:", len(missing))
display(delta_counts.head(5))

## 3. Forbidden-Column Check

**Purpose:** Version 1 is external-forcing only. It should not contain `influent_flow_mgd`, `flow_*` lag columns, model predictions, or any synthetic target. Those belong later, after real hourly North River influent-flow data is available.

In [ ]:
# V1 should only contain external drivers, not target or prediction columns.
forbidden = [
    col
    for col in df.columns
    if col == "influent_flow_mgd"
    or col.startswith("flow_")
    or "prediction" in col.lower()
    or "pred" in col.lower()
]
print("forbidden columns:", forbidden)

## 4. Missing-Data Summary

**Purpose:** Missing rainfall or tide values are allowed, but they need to be visible and flagged. The goal is not to hide missingness by silently filling everything with zero.

In [ ]:
# Compare raw null counts with the explicit missing-data flags.
check_cols = ["rain_mm", "water_level", "rain_missing_flag", "tide_missing_flag"]
for col in check_cols:
    if col in df.columns:
        nulls = df.select(pl.col(col).null_count()).item()
        print(f"{col} null count: {nulls}")

for flag in ["rain_missing_flag", "tide_missing_flag"]:
    if flag in df.columns:
        print(f"{flag} sum:", df.select(pl.col(flag).sum()).item())

## 5. Rainfall Sanity Checks

**Purpose:** Rainfall should be sparse and storm-driven: many zero hours, occasional spikes, and rolling rainfall features that smooth or accumulate storm effects.

The storm-window plot below uses the remnants of Hurricane Ida in NYC, roughly `2021-09-01` through `2021-09-03`. Ida produced extreme rainfall in NYC, so it is a useful stress-test window for visual inspection. This is only a sanity check on the feature store, not proof that any model is correct.

In [ ]:
# Check the overall rainfall distribution before looking at a specific storm.
rain_summary = df.select(
    pl.col("rain_mm").min().alias("min_rain_mm"),
    pl.col("rain_mm").max().alias("max_rain_mm"),
    pl.col("rain_mm").mean().alias("mean_rain_mm"),
    (pl.col("rain_mm") > 0).sum().alias("rainy_hours"),
)
display(rain_summary)

largest_rain = df.select("timestamp", "rain_mm", "rain_last_24h").sort("rain_mm", descending=True).head(10)
display(largest_rain)

In [ ]:
# Hurricane Ida remnants: a known high-rainfall NYC window for visual stress testing.
storm_start = pd.Timestamp("2021-09-01")
storm_end = pd.Timestamp("2021-09-04")  # includes 2021-09-01 through 2021-09-03
storm_pd = (
    df.filter((pl.col("timestamp") >= storm_start) & (pl.col("timestamp") <= storm_end))
    .select("timestamp", "rain_mm", "rain_last_24h")
    .to_pandas()
)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(storm_pd["timestamp"], storm_pd["rain_mm"], marker="o", linewidth=1)
axes[0].set_title("Rainfall over sample storm window")
axes[0].set_ylabel("rain_mm")
axes[1].plot(storm_pd["timestamp"], storm_pd["rain_last_24h"], color="tab:green")
axes[1].set_title("Prior 24-hour rainfall over sample storm window")
axes[1].set_ylabel("rain_last_24h")
plt.tight_layout()

## 6. Tide Sanity Checks

**Purpose:** Tide should show periodic water-level movement. The Battery is used here as a proxy for lower Hudson / New York Harbor tidal conditions near Manhattan, not as a plant-specific hydraulic boundary condition.

This section also checks that the synthetic tidal phase features stay between `-1` and `1`, as sine and cosine features should.

In [ ]:
# Water level should be in a plausible range, and sine/cosine phase features should stay bounded.
tide_summary = df.select(
    pl.col("water_level").min().alias("min_water_level"),
    pl.col("water_level").max().alias("max_water_level"),
    pl.col("water_level").mean().alias("mean_water_level"),
    pl.col("tidal_phase_sin").min().alias("min_tidal_phase_sin"),
    pl.col("tidal_phase_sin").max().alias("max_tidal_phase_sin"),
    pl.col("tidal_phase_cos").min().alias("min_tidal_phase_cos"),
    pl.col("tidal_phase_cos").max().alias("max_tidal_phase_cos"),
)
display(tide_summary)

In [ ]:
# A normal 7-day window should show the repeated tidal cycle clearly.
tide_start = pd.Timestamp("2022-06-01")
tide_end = pd.Timestamp("2022-06-08")
tide_pd = (
    df.filter((pl.col("timestamp") >= tide_start) & (pl.col("timestamp") < tide_end))
    .select("timestamp", "water_level", "water_level_change_1h")
    .to_pandas()
)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(tide_pd["timestamp"], tide_pd["water_level"])
axes[0].set_title("Battery water level over representative 7-day period")
axes[0].set_ylabel("meters")
axes[1].plot(tide_pd["timestamp"], tide_pd["water_level_change_1h"], color="tab:orange")
axes[1].set_title("1-hour water-level change")
axes[1].set_ylabel("meters/hour")
plt.tight_layout()

## 7. Lag and Rolling Feature Spot Checks

**Purpose:** These checks make sure engineered features are causal. For example, `rain_t_1` should come from the previous hour, `rain_t_6` from six hours earlier, and `rain_t_24` from 24 hours earlier. Rolling rainfall should use prior rainfall, not future rainfall. For tide, `water_level_change_1h` should equal current water level minus the previous hour's water level.

In [ ]:
# Spot-check a few timestamps instead of dumping the full dataset.
sample_times = [
    pd.Timestamp("2021-09-02 03:00"),
    pd.Timestamp("2022-06-01 12:00"),
    pd.Timestamp("2023-12-18 06:00"),
]

for ts in sample_times:
    row = df.filter(pl.col("timestamp") == ts)
    if row.height == 0:
        print("missing sample timestamp:", ts)
        continue
    prev_1 = df.filter(pl.col("timestamp") == ts - pd.Timedelta(hours=1))
    prev_6 = df.filter(pl.col("timestamp") == ts - pd.Timedelta(hours=6))
    prev_24 = df.filter(pl.col("timestamp") == ts - pd.Timedelta(hours=24))
    # Prior 24 hours excludes the current hour so this remains causal.
    prior_24 = df.filter((pl.col("timestamp") >= ts - pd.Timedelta(hours=24)) & (pl.col("timestamp") < ts))

    print("\nTimestamp:", ts)
    print("rain_t_1 equals previous rain_t:", row["rain_t_1"][0], prev_1["rain_t"][0] if prev_1.height else None)
    print("rain_t_6 equals rainfall 6 hours earlier:", row["rain_t_6"][0], prev_6["rain_mm"][0] if prev_6.height else None)
    print("rain_t_24 equals rainfall 24 hours earlier:", row["rain_t_24"][0], prev_24["rain_mm"][0] if prev_24.height else None)
    if prior_24["rain_mm"].null_count() == 0 and prior_24.height == 24:
        print("rain_last_24h vs prior 24-hour sum:", row["rain_last_24h"][0], prior_24["rain_mm"].sum())
    else:
        print("rain_last_24h check skipped because prior 24 hours contain missing rainfall")
    print("water_level_t_1 equals previous water level:", row["water_level_t_1"][0], prev_1["water_level"][0] if prev_1.height else None)
    print("water_level_change_1h check:", row["water_level_change_1h"][0], row["water_level"][0] - row["water_level_t_1"][0] if row["water_level_t_1"][0] is not None else None)

## 8. Storm and Antecedent Feature Sanity Checks

**Purpose:** These features are meant to capture wet-weather memory. Storm duration and storm total should reset during dry hours. Antecedent wetness indexes should rise after rain and decay over time.

In [ ]:
# Dry hours should not keep accumulating storm duration or storm total.
storm_cols = ["storm_duration", "storm_total_rainfall", "hours_since_last_rain", "AWI_3d", "AWI_7d"]
available_storm_cols = [col for col in storm_cols if col in df.columns]
display(df.select(["timestamp", "rain_mm"] + available_storm_cols).filter(pl.col("rain_mm") == 0).head(10))

dry_nonzero = df.filter((pl.col("rain_mm") == 0) & ((pl.col("storm_duration") != 0) | (pl.col("storm_total_rainfall") != 0)))
print("dry hours with nonzero storm duration/total:", dry_nonzero.height)

In [ ]:
# Ida month is useful for checking whether antecedent wetness rises after rain and then decays.
awi_start = pd.Timestamp("2021-09-01")
awi_end = pd.Timestamp("2021-10-01")
awi_pd = (
    df.filter((pl.col("timestamp") >= awi_start) & (pl.col("timestamp") < awi_end))
    .select("timestamp", "rain_mm", "AWI_3d", "AWI_7d")
    .to_pandas()
)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(awi_pd["timestamp"], awi_pd["AWI_3d"], label="AWI_3d")
ax.plot(awi_pd["timestamp"], awi_pd["AWI_7d"], label="AWI_7d")
ax.set_title("Antecedent wetness over stormy month")
ax.legend()
plt.tight_layout()

## 9. Source Metadata Check

**Purpose:** Source labels, units, and datum columns are included so the feature store keeps its provenance. This makes it harder to accidentally mix rainfall or water-level sources later without noticing.

In [ ]:
# These values should be stable and easy to read in the output table.
for col in ["rain_source", "tide_source", "water_level_units", "water_level_datum"]:
    print(col)
    display(df.select(col).unique())

## 10. Plain-English Conclusion Checklist

Passing this notebook means the feature store is structurally sound: it has the expected time axis, rainfall and tide were joined, feature engineering looks plausible, missingness is visible, and forbidden target/flow columns are absent.

It does **not** prove model quality because no target-flow model exists yet. The modeling step has to wait until real hourly North River influent-flow data is available.

Checklist:

- Complete hourly backbone?
- Rainfall joined?
- Tide joined?
- Calendar features present?
- Rainfall lags present?
- Rolling rainfall features present?
- Tide features present?
- Tidal phase features present and bounded between `-1` and `1`?
- Missing flags present?
- Source metadata present?
- Forbidden target/flow columns absent?
- Known limitations noted?

Known limitations to keep in mind: Central Park rainfall is a point-gauge fallback, The Battery is a tide proxy, V1 uses naive local timestamps, and real hourly North River influent-flow data is still needed before modeling.